In [ ]:
"""Day 20 / Session 2 - Lab Assignment 2: Saliency Maps and Occlusion Sensitivity - Two Ways to See What a CNN Sees."""
import numpy as np


def saliency_map(grad):
    return np.abs(np.asarray(grad, dtype=float)).max(axis=0)      # |gradient|, then max over channels


def normalize01(x):
    x = np.asarray(x, dtype=float)
    lo, hi = x.min(), x.max()
    if hi == lo:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def occlude(image, top, left, size, fill=0.0):
    out = np.array(image, dtype=float, copy=True)                 # never mutate the input
    out[top:top + size, left:left + size] = fill
    return out


def occlusion_map(image, score_fn, size, stride, fill=0.0):
    image = np.asarray(image, dtype=float)
    H, W = image.shape
    baseline = score_fn(image)
    rows = list(range(0, H - size + 1, stride))
    cols = list(range(0, W - size + 1, stride))
    grid = np.zeros((len(rows), len(cols)))
    for a, top in enumerate(rows):
        for b, left in enumerate(cols):
            grid[a, b] = baseline - score_fn(occlude(image, top, left, size, fill))   # big drop = important region
    return grid
